# Day 3: HuggingFace 生态探索

本节学习 HuggingFace 三大核心库：**transformers**、**datasets**、**tokenizers**

学习目标：
1. 理解 HuggingFace 生态系统
2. 深入理解 Tokenizer 的工作原理
3. 掌握预训练模型的加载和使用
4. 学会使用 Datasets 库进行数据处理
5. 用 HF Trainer 完成一次文本分类微调

## 1. HuggingFace 生态介绍

HuggingFace 提供了三大核心库：

| 库 | 功能 | 核心类 |
|---|------|--------|
| **transformers** | 模型加载、训练、推理 | AutoModel, Trainer |
| **datasets** | 数据集加载与处理 | load_dataset, Dataset |
| **tokenizers** | 快速文本分词 | AutoTokenizer |

此外还有：
- **peft**: 参数高效微调（LoRA 等）
- **trl**: 强化学习训练（SFT, DPO, PPO）
- **accelerate**: 分布式训练
- **Hub**: 模型/数据集托管平台

## 2. Tokenizer 深入分析

### 2.1 加载 Qwen2-0.5B Tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2-0.5B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Tokenizer 类型: {type(tokenizer).__name__}")
print(f"词表大小 (vocab_size): {tokenizer.vocab_size}")
print(f"模型最大长度: {tokenizer.model_max_length}")

### 2.2 Special Tokens 分析

Special tokens 是模型用来标识特殊语义的 token，如句子开头、结尾、填充等。

In [ ]:
# 查看特殊 tokens
print("特殊 Tokens:")
print(f"  PAD token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")
print(f"  EOS token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"  BOS token: {tokenizer.bos_token} (id: {tokenizer.bos_token_id})")
print(f"  UNK token: {tokenizer.unk_token} (id: {tokenizer.unk_token_id})")

# 查看所有特殊 tokens
print(f"\n所有特殊 tokens: {tokenizer.all_special_tokens}")
print(f"对应 IDs: {tokenizer.all_special_ids}")

### 2.3 Encode / Decode 演示

Tokenizer 的核心功能：将文本转换为 token ID 序列（encode），以及反向操作（decode）。

In [ ]:
# 编码解码测试
test_texts = [
    "Hello, world!",
    "你好，世界！",
    "机器学习是人工智能的一个分支。",
]

for text in test_texts:
    tokens = tokenizer.encode(text)
    decoded = tokenizer.decode(tokens)
    token_strs = tokenizer.convert_ids_to_tokens(tokens)
    print(f"原文: {text}")
    print(f"Token IDs ({len(tokens)} tokens): {tokens}")
    print(f"Token 字符串: {token_strs}")
    print(f"解码: {decoded}")
    print()

In [ ]:
# 对比中英文 tokenization 效率
en_text = "Machine learning is a branch of artificial intelligence."
zh_text = "机器学习是人工智能的一个分支。"

en_tokens = tokenizer.encode(en_text)
zh_tokens = tokenizer.encode(zh_text)

print(f"英文: {len(en_text)} 字符 -> {len(en_tokens)} tokens")
print(f"中文: {len(zh_text)} 字符 -> {len(zh_tokens)} tokens")
print(f"\n中文/英文 token 比: {len(zh_tokens)/len(en_tokens):.2f}")

### 2.4 Chat Template

现代 LLM 使用 Chat Template 来格式化多轮对话，区分不同角色（system、user、assistant）。

In [ ]:
# 查看 Chat Template
if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
    print("Chat Template:")
    print(tokenizer.chat_template[:500])
else:
    print("该模型没有 chat template")

In [ ]:
# 应用 Chat Template
messages = [
    {"role": "system", "content": "你是一个有用的助手。"},
    {"role": "user", "content": "什么是机器学习？"},
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False)
print("格式化后的对话:")
print(formatted)

# 加上 generation prompt
formatted_gen = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("\n加 generation prompt:")
print(formatted_gen)

## 3. 预训练模型加载与使用

### 3.1 加载 Qwen2-0.5B 模型

In [ ]:
# 加载因果语言模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")

### 3.2 查看模型结构细节

In [ ]:
# 查看各层参数量
print("模型结构概览:")
for name, param in model.named_parameters():
    if "layers.0." in name or "embed" in name or "lm_head" in name:
        print(f"  {name}: {param.shape} ({param.numel():,})")

# 模型配置
print(f"\n模型配置:")
print(f"  隐藏层维度: {model.config.hidden_size}")
print(f"  注意力头数: {model.config.num_attention_heads}")
print(f"  层数: {model.config.num_hidden_layers}")
print(f"  词表大小: {model.config.vocab_size}")

### 3.3 文本生成演示

In [ ]:
# 用模型生成文本
prompt = "人工智能的未来发展趋势是"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        top_k=50,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"生成: {generated}")

## 4. Datasets 库

### 4.1 加载数据集

In [ ]:
from datasets import load_dataset
import numpy as np

# 加载 Yelp 评论数据集
dataset = load_dataset("yelp_review_full", trust_remote_code=True)
print(f"数据集结构: {dataset}")
print(f"训练集大小: {len(dataset['train'])}")
print(f"\n样本示例:")
print(dataset["train"][0])

### 4.2 数据预处理 Pipeline

`datasets` 库提供了高效的 `map`、`filter`、`select` 等操作。

In [ ]:
# select: 取子集
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))
print(f"取子集: 训练 {len(small_train)}, 测试 {len(small_test)}")

# filter: 过滤
five_star = small_train.filter(lambda x: x["label"] == 4)
print(f"5星评论数: {len(five_star)}")

# map: 批量处理
def add_text_length(example):
    example["text_length"] = len(example["text"])
    return example

small_train = small_train.map(add_text_length)
print(f"\n文本长度统计:")
lengths = small_train["text_length"]
print(f"  平均: {np.mean(lengths):.0f}, 最短: {min(lengths)}, 最长: {max(lengths)}")

In [ ]:
# Tokenize 预处理
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_train = small_train.map(preprocess_function, batched=True, remove_columns=["text", "text_length"])
tokenized_test = small_test.map(preprocess_function, batched=True, remove_columns=["text"])

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print(f"Tokenize 完成!")
print(f"训练集: {len(tokenized_train)}, 测试集: {len(tokenized_test)}")
print(f"特征: {tokenized_train.column_names}")

## 5. HF Trainer 文本分类微调

### 5.1 加载分类模型 & 微调前预测

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

NUM_LABELS = 5  # Yelp: 1-5 星

model_cls = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    trust_remote_code=True,
    torch_dtype=torch.float32,
)
model_cls.config.pad_token_id = tokenizer.pad_token_id
model_cls.to(DEVICE)
model_cls.eval()

# 微调前预测
test_texts_cls = [
    "This restaurant is absolutely amazing! Best food ever!",
    "Terrible experience. The food was cold and the service was awful.",
    "It was okay, nothing special but not bad either.",
]

print("微调前的预测（随机分类头）:")
for text in test_texts_cls:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        outputs = model_cls(**inputs)
    pred = outputs.logits.argmax(dim=-1).item()
    print(f"  {text[:50]}... -> {pred + 1} 星")

### 5.2 配置 Trainer 并训练

In [ ]:
import time

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="./hf_trainer_output",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model_cls,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("开始训练...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time
print(f"训练完成! 耗时: {elapsed:.0f}s")
print(f"训练 Loss: {train_result.metrics['train_loss']:.4f}")

### 5.3 评估 & 微调前后对比

In [ ]:
# 评估
eval_result = trainer.evaluate()
print(f"测试集准确率: {eval_result['eval_accuracy']:.4f}")

# 微调后预测
print("\n微调后的预测:")
model_cls.eval()
for text in test_texts_cls:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        outputs = model_cls(**inputs)
    pred = outputs.logits.argmax(dim=-1).item()
    print(f"  {text[:50]}... -> {pred + 1} 星")

## 6. 总结

### 今日要点

1. **Tokenizer**: 词表大小、special tokens、chat template 是理解 LLM 输入的基础
2. **模型加载**: `AutoModelForCausalLM` 适合生成任务，`AutoModelForSequenceClassification` 适合分类
3. **Datasets**: `map`/`filter`/`select` 构成高效的数据处理 pipeline
4. **Trainer**: HuggingFace 的高层训练 API，封装了训练循环、评估、保存等逻辑

### 思考题

1. 为什么 Qwen2 的 tokenizer 对中文和英文的 token 效率不同？
2. Chat Template 的作用是什么？如果不用 chat template 直接输入文本会怎样？
3. `map(batched=True)` 和 `map(batched=False)` 的性能差异来自哪里？
4. Trainer 的 `load_best_model_at_end=True` 是如何工作的？